# Setup

In [4]:
from bitarray import bitarray
def rand10(w,k): #computes the expected density of random 10-order (w>=k-2)
    #calls dfs10pref, dfs10no
    global probab, kmers
    kmers = bitarray(1<<k-2)
    probab = [0 for i in range(w//2+1)]    #probab[i] refers to windows with i+1 distinct 10-k-mers
    for smer in range(1<<k-2):  #loop through 10-prefixes
        dfs10pref(smer,0,w,k,1)
    for i in range(1,w-1): #first 10-kmer at position i (i=w-1 skipped as never charged)
        for smer in range(1<<k-2):  #loop through 10-kmers
            dfs10no(smer,i,w,k,i+1)
    probab[0] += (w+1)*(1<<k-2) #add windows having k-suffix as the only 10-kmer
    print(probab)
    probsum = probab[0]
    for ii in range(1,len(probab)):
        probsum += probab[ii] / (ii+1)
    return (w+1) * (probsum + w+k+3 + (w==k-2)) / (1<<w+k)     # density factor

def dfs10pref(smer,dep,w,k,multi): #auxiliary# recursively counts distinct 10-kmers in windows with 10-prefix
    length = (smer & ~(smer<<1)).bit_length() #longest suffix of smer where an 10-kmer begins
    shift = k - length
    flag = kmers[smer]
    kmers[smer] = 1
    lo = kmers.count(1) #new number of 10-kmers
    if dep+shift>w:     #processed 10-kmer is the last
        probab[lo-1] += multi*(1<<w-dep)  #lo 10-k-mers, not suff-charged
    elif dep+shift==w:  #the next possible 10-kmer is k-suffix: adding #windows to probab
        if length>1:    #k-suffix is 10-kmer
            tail = (smer & (1<<length-2)-1)<<shift #known prefix of the k-suffix padded with 0s
            replo = kmers[tail:tail+(1<<shift)].count(1) #number of non-fresh k-suffixes 
            probab[lo-1] += multi*replo #not suff-charged
            probab[lo] += 2*multi*((1<<shift) - replo)    #maybe suff-charged
        elif length==1: #k-suffix = 1...., can be ANY 10-kmer
            probab[lo-1] += multi*((1<<k-2) + lo)   #k-suffix is not 10-kmer or old; not suff-charged window
            probab[lo] += 2*multi*((1<<k-2) - lo)   #new 10-k-suffix, window maybe suff-charged
        else: #k-suffix = ?????
            probab[lo-1] += multi*((3<<k-2) + lo)   #k-suffix is not 10-kmer or old; not suff-charged window
            probab[lo] += 2*multi*((1<<k-2) - lo)   #new 10-k-suffix, window maybe suff-charged
    else:   #recursive call
        if length>1:    #next 10-kmer position is inside smer
            tail = (smer & (1<<length-2)-1)<<shift #known prefix of the next 10-kmer padded with 0s
            for i in range(1<<shift):
                dfs10pref(tail+i,dep+shift,w,k,multi)  #process (k-2)-mer following 10
        elif length==1: #smer = (0..0)1..1
            for i in range(dep+shift,w): #position of the next 10-kmer (if <w)
                for tail in range(1<<k-2):  #for all possible 10-kmers
                    dfs10pref(tail,i,w,k,multi)
            probab[lo-1] += multi*((1<<k-2) + lo)   #no more 10-kmers or "old" 10-k-suffix
            probab[lo] += 2*multi*((1<<k-2) - lo)   #"new" 10-k-suffix
        else:  #smer = 0..0
            for i in range(dep+shift,w): #position of the next 10-kmer (if <w)
                for tail in range(1<<k-2):  #for all possible 10-kmers
                    dfs10pref(tail,i,w,k,multi*(i-shift-dep+1)) #flexible between 0..0 and next 10
            probab[lo-1] += multi*((w-dep-shift+3)*(1<<k-2) + (w-dep-shift+1)*lo) #no more 10-kmers or "old" 10-k-suffix
            probab[lo] += 2*multi*(w-dep-shift+1)*((1<<k-2) - lo) #"new" 10-k-suffix
    kmers[smer]=flag    #restore original state of kmers

def dfs10no(smer,dep,w,k,multi): #auxiliary# recursively counts distinct 10-kmers in windows without 10-prefix
    length = (smer & ~(smer<<1)).bit_length() #longest suffix of smer where an 10-kmer begins
    shift = k - length
    flag = kmers[smer]
    kmers[smer] = 1
    lo = kmers.count(1) #new number of 10-kmers
    if dep+shift==w:  #the next possible 10-kmer is k-suffix: adding #windows to probab
        if length>1:    #k-suffix is 10-kmer
            tail = (smer & (1<<length-2)-1)<<shift #known prefix of the k-suffix padded with 0s
            replo = kmers[tail:tail+(1<<shift)].count(1) #number of non-fresh k-suffixes 
            probab[lo] += multi*((1<<shift) - replo)    #new 10-k-suffix, window maybe suff-charged
        else: #k-suffix can be any 10-kmer
            probab[lo] += multi*((1<<k-2) - lo)   #new 10-k-suffix, window maybe suff-charged
    elif dep+shift<w:   #recursive call ##no charged windows if dep+shift>w
        if length>1:    #next 10-kmer position is inside smer
            tail = (smer & (1<<length-2)-1)<<shift #known prefix of the next 10-kmer padded with 0s
            for i in range(1<<shift):
                dfs10no(tail+i,dep+shift,w,k,multi)  #process (k-2)-mer following 10
        elif length==1: #smer = (0..0)1..1
            for i in range(dep+shift,w): #position of the next 10-kmer (if <w)
                for tail in range(1<<k-2):  #for all possible 10-kmers
                    dfs10no(tail,i,w,k,multi)
            probab[lo] += multi*((1<<k-2) - lo)   #"new" 10-k-suffix
        else:  #smer = 0..0
            for i in range(dep+shift,w): #position of the next 10-kmer (if <w)
                for tail in range(1<<k-2):  #for all possible 10-kmers
                    dfs10no(tail,i,w,k,multi*(i-shift-dep+1)) #flexible between 0..0 and next 10
            probab[lo] += multi*(w-dep-shift+1)*((1<<k-2) - lo) #"new" 10-k-suffix
    kmers[smer]=flag    #restore original state of kmers


# Expriments

We show an example experiment where k=7 and 5<=w<=14.

In [15]:
k=7
for w in range(5, 15):
    print(f'density factor for w={w}, k={k}: {rand10(w,k)}')

[404, 1262, 352]
density factor for w=5, k=7: 1.71142578125
[476, 2255, 1260, 32]
density factor for w=6, k=7: 1.74957275390625
[574, 3644, 3404, 392]
density factor for w=7, k=7: 1.7801106770833335
[659, 5641, 7732, 1936, 0]
density factor for w=8, k=7: 1.8014373779296877
[800, 8238, 15758, 6712, 304]
density factor for w=9, k=7: 1.8202921549479167
[906, 11841, 29514, 19012, 2112, 0]
density factor for w=10, k=7: 1.8345558166503908
[1182, 16286, 52093, 46490, 10084, 144]
density factor for w=11, k=7: 1.8481582641601562
[1454, 22309, 87592, 102949, 35628, 1668, 0]
density factor for w=12, k=7: 1.8588884035746256
[1842, 30526, 141174, 211852, 104670, 11228, 0]
density factor for w=13, k=7: 1.8685919443766277
[2366, 41612, 221618, 409386, 272227, 50744, 844, 0]
density factor for w=14, k=7: 1.8771062237875804
